In [6]:
import pandas as pd
import polars as pl
import numpy as np
import gc
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier

In [9]:
train_part1 = pl.read_parquet("../data/test.parquet")

In [10]:
train_part1 = train_part1.with_columns([
    pl.col("compromised").cast(pl.Int32),
    pl.col("developer_tools").cast(pl.Int32),
    pl.col("mcc_code").cast(pl.Int32)
])

In [11]:
train_part1.schema

Schema([('customer_id', Int64),
        ('event_id', Int64),
        ('event_dttm', String),
        ('event_type_nm', Int32),
        ('event_desc', Int32),
        ('channel_indicator_type', Int32),
        ('channel_indicator_sub_type', Int32),
        ('operaton_amt', Float64),
        ('currency_iso_cd', Int32),
        ('mcc_code', Int32),
        ('pos_cd', Int32),
        ('accept_language', String),
        ('browser_language', String),
        ('timezone', Int32),
        ('session_id', Int64),
        ('operating_system_type', Int32),
        ('battery', String),
        ('device_system_version', String),
        ('screen_size', String),
        ('developer_tools', Int32),
        ('phone_voip_call_state', Int32),
        ('web_rdp_connection', Int32),
        ('compromised', Int32)])

In [12]:
train_part1 = train_part1.with_columns(pl.col("event_dttm").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S")\
                                 .alias("event_dttm")).with_columns(pl.col("event_dttm").dt.hour().alias("Hour"))

In [13]:
train_part1 = train_part1.sort("event_dttm")
train_part1 = train_part1.drop("event_dttm")

In [ ]:
cat_features = [
    'mcc_code', 'event_desc',
    'timezone', 'operating_system_type', 'device_system_version',
    'screen_size', 'battery'
]

for i in cat_features:
    train_part1 = train_part1.with_columns(pl.col(i).fill_null(-1))

In [15]:
delete = ["accept_language", "browser_language"]

train_part1 = train_part1.drop(delete)

In [16]:
train_part1 = train_part1.drop("customer_id")

In [17]:
event_id = train_part1["event_id"].to_pandas()
train_part1 = train_part1.drop("event_id")

In [18]:
train_part1

event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,Hour
i32,i32,i32,i32,f64,i32,i32,i32,i32,i64,i32,str,str,str,i32,i32,i32,i32,i8
8,122,2,0,null,null,-1,null,-1,126121010775326,-1,"""-1""","""-1""","""-1""",null,null,null,null,0
7,56,3,4,null,null,-1,null,16,124214045774300,9,"""99%""","""-1""","""-1""",null,null,0,null,0
14,75,0,5,85864.0,0,19,null,-1,null,-1,"""-1""","""-1""","""-1""",null,null,null,null,0
14,75,0,5,44216.0,0,4,null,-1,null,-1,"""-1""","""-1""","""-1""",null,null,null,null,0
2,88,0,5,null,0,10,3,-1,null,-1,"""-1""","""-1""","""-1""",null,null,null,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7,56,3,4,null,null,-1,null,42,123818908925253,6,"""not available""","""-1""","""-1""",null,null,0,null,23
8,122,2,0,null,null,-1,null,-1,126318579340682,-1,"""-1""","""-1""","""-1""",null,null,null,null,23
14,75,0,5,65121.0,0,4,3,-1,null,-1,"""-1""","""-1""","""-1""",null,null,null,null,23


Получили обработанный набор данных, теперь можно переходить прогнозированию.

In [19]:
model = CatBoostClassifier()
model.load_model('../Models/Model_PC_5.1.cbm')

CatBoostClassifier(class_names=[0, 1], class_weights=[1, 100], depth=5, iterations=200, loss_function='Logloss', verbose=0)

In [20]:
predict = model.predict(train_part1)

In [21]:
y_pred_proba = model.predict_proba(train_part1)[:, 1]
y_pred_proba

array([0.00823219, 0.40880978, 0.63318052, ..., 0.24125987, 0.34282199,
       0.1737676 ], shape=(633683,))

In [22]:
y_pred_proba = pd.Series(y_pred_proba, name="predict")
y_pred_proba

0         0.008232
1         0.408810
2         0.633181
3         0.245966
4         0.313912
            ...   
633678    0.198537
633679    0.000555
633680    0.241260
633681    0.342822
633682    0.173768
Name: predict, Length: 633683, dtype: float64

In [23]:
event_id = pd.Series(event_id, name="event_id")

In [24]:
submit = pd.concat([event_id, y_pred_proba], names=["event_id", "predict"], axis=1)

In [25]:
submit

,event_id,predict
0,125339330014816,0.008232
1,124978549756126,0.408810
2,126198320503253,0.633181
3,125390866897300,0.245966
4,123879040110074,0.313912
...,...,...
633678,126146783971142,0.198537
633679,123999297165929,0.000555
633680,123561209857910,0.241260
633681,125674336254442,0.342822


In [26]:
submit.to_csv("submit.csv", index=False)